![WEkEO Banner](https://github.com/wekeo/frameworks/blob/main/img/WEkEO_banner_partners.png?raw=true)

<div style="display: flex; justify-content: space-between;">
  <a href="./README.md">&lt;&lt; README</a>
  <a href="./DASK_Workers_Processing_Result.ipynb">DASK Gateway Performance Comparison &gt;&gt;</a>
</div>

Copyright (c) 2025 European Union 

License: MIT

# Introduction to DASK and the WEkEO DASK Gateway

**This notebook can only be run on the WEkEO JupyterHub platform.** 

[![launch-WEKEO](https://img.shields.io/badge/launch-WEKEO-1a4696.svg)](https://jupyterhub.prod.wekeo2.eu/TOBEFILLED)


**PREREQUISITES**

This notebook has the following prerequisites:
- [A WEkEO account](https://my.wekeo.eu/user-registration) 
- Access to the WEkEO JupyterHub workspace
- Basic knowledge of Python and Jupyter notebooks

**LEARNING OUTCOMES**

At the end of this notebook you will know: 

- What DASK is and why it's useful for data science
- When to use DASK for your data processing tasks
- How to connect to the WEkEO DASK Gateway
- How to create and manage DASK clusters on WEkEO
- How to perform parallel computations using DASK arrays

**CONTENTS**

1. [What is DASK?](#1-what-is-dask)  
2. [Why and when to use DASK?](#2-why-and-when-to-use-dask)  
3. [DASK on WEkEO: The DASK Gateway](#3-dask-on-wekeo-the-dask-gateway)  
4. [Getting started with the DASK Gateway](#4-getting-started-with-the-dask-gateway)  
5. [Your first parallel computation](#5-your-first-parallel-computation)  
6. [Managing your DASK cluster](#6-managing-your-dask-cluster)  
7. [Take home messages](#7-take-home-messages)

<hr>

## 1. What is DASK?

[**DASK**](https://www.dask.org/) is a flexible parallel computing library for Python that enables you to scale your data analysis workflows from your laptop to large clusters. It provides parallel and distributed computing capabilities while using familiar Python APIs like NumPy, Pandas, and scikit-learn.

### Key features of DASK:

- **Parallel computing**: DASK breaks down large computations into smaller tasks that can be executed in parallel across multiple cores or machines
- **Lazy evaluation**: Operations are not executed immediately but are recorded as a task graph, which is only executed when you explicitly request results
- **Familiar APIs**: DASK provides drop-in replacements for NumPy arrays (`dask.array`), Pandas DataFrames (`dask.dataframe`), and more
- **Scales from laptops to clusters**: The same code can run on your local machine or be scaled to a large distributed cluster
- **Memory efficiency**: DASK works with datasets larger than your available RAM by processing data in chunks

For more detailed information, visit the [official DASK documentation](https://docs.dask.org/).

## 2. Why and when to use DASK?

### When DASK is useful:

✅ **Large datasets**: When your data doesn't fit into memory (larger than your available RAM)

✅ **Parallel processing**: When you want to speed up computations by utilizing multiple CPU cores

✅ **Array operations**: Working with large multi-dimensional arrays (e.g., climate data, satellite imagery)

✅ **Scalability**: When you need the flexibility to scale from local development to cloud computing

The WEkEO Workspace DASK Gateway is a good starting point to **make your code DASK-ready** before moving to larger cloud systems to scale up data processing. 

### When DASK might not be necessary:

❌ **Small datasets**: Data that easily fits in memory and processes quickly with standard libraries

❌ **I/O bound tasks**: Operations primarily limited by disk or network speed rather than computation

❌ **Complex dependencies**: Tasks with intricate dependencies between operations may not benefit from parallelization

❌ **Simple operations**: Basic data analysis that runs quickly without optimization

### Common use cases while working with Coperncius Data on WEkEO:

- Processing large satellite image archives
- Climate data analysis with multi-dimensional arrays (using xarray + DASK)
- Time-series analysis of environmental data
- Large-scale geospatial computations
- Machine learning on large datasets

## 3. DASK on WEkEO: The DASK Gateway

The **WEkEO DASK Gateway** provides a secure and managed way to create and use DASK clusters on the WEkEO infrastructure. 

### What is a DASK Gateway?

A DASK Gateway is a multi-user, multi-cluster service that provides a way to connect your processing on the WEkEO Workspace to additional resources in the form of DASK workers.

When you use the WEkEO DASK Gateway, you work with:

1. **Scheduler**: Coordinates the work across workers
2. **Workers**: Execute the actual computations in parallel
3. **Client**: Your Jupyter notebook connects to the cluster through a client object

```
┌─────────────────┐
│ Your Notebook   │
│   (Client)      │
└────────┬────────┘
         │
    ┌────▼────┐
    │Scheduler│
    └────┬────┘
         │
    ┌────┴────┐
    │         │
┌───▼──┐  ┌──▼───┐
│Worker│  │Worker│  ...
└──────┘  └──────┘
```

## 4. Getting started with the DASK Gateway

### 4.1 Import required libraries

Let's start by importing the necessary libraries for working with the DASK Gateway.

In [1]:
# Import DASK Gateway libraries
from dask_gateway import Gateway, GatewayCluster

# Import DASK and related libraries
import dask.array as da
import xarray as xr
import numpy as np

# For displaying cluster information
from IPython.display import display

### 4.2 Connect to the WEkEO DASK Gateway

First, we establish a connection to the WEkEO DASK Gateway.

In [2]:
# Create a gateway connection
gateway = Gateway()

print("Successfully connected to the WEkEO DASK Gateway!")
print(f"Gateway address: {gateway.address}")

Successfully connected to the WEkEO DASK Gateway!
Gateway address: http://proxy-public/services/dask-gateway


### 4.3 Configure cluster options

Before creating a cluster, we can configure the resources for our workers. On WEkEO, you can specify:

- **Worker cores**: Number of CPU cores per worker
- **Worker memory**: Amount of RAM per worker (in GB)

In [3]:
# Get the available cluster options
options = gateway.cluster_options()

# Display the options interface (in Jupyter, this shows a widget)
display(options)

Let's configure our cluster with specific resource settings:

In [4]:
# Set worker resources
options.worker_cores = 1      # 1 CPU core per worker
options.worker_memory = 1.0   # 1 GB RAM per worker

print(f"Cluster configured with:")
print(f"  - Worker cores: {options.worker_cores}")
print(f"  - Worker memory: {options.worker_memory} GB")

Cluster configured with:
  - Worker cores: 1
  - Worker memory: 1.0 GB


### 4.4 Create and scale your DASK cluster

Now we can create a cluster with our specified configuration.

In [5]:
# Create a new cluster
cluster = GatewayCluster(cluster_options=options)

# Scale the cluster to 2 workers
cluster.scale(2)

print(f"Cluster created!")
print(f"Cluster name: {cluster.name}")

# Display the cluster (shows a dashboard link and status)
display(cluster)

Cluster created!
Cluster name: ws-jupyterhub.b1a011d989384f12b3b6c3ea81618554


:::{note}

The cluster dashboard provides a real-time view of your DASK cluster's performance. You can monitor:
- Task progress and completion
- Memory usage across workers
- Worker status and activity
- Task graphs and execution timelines

Click on the dashboard link above to open it in a new tab!

:::

### 4.5 Connect a client to the cluster

To actually use the cluster, we need to create a client that connects our notebook to it.

In [6]:
# Get a client connected to the cluster
client = cluster.get_client()

print("Client connected to cluster!")
print(f"Dashboard link: {client.dashboard_link}")

# Display client information
display(client)

Client connected to cluster!
Dashboard link: /services/dask-gateway/clusters/ws-jupyterhub.b1a011d989384f12b3b6c3ea81618554/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1393: VersionMismatchWarning: Mismatched versions found

+---------+-----------------+----------------+---------+
| Package | Client          | Scheduler      | Workers |
+---------+-----------------+----------------+---------+
| python  | 3.11.11.final.0 | 3.11.9.final.0 | None    |
| tornado | 6.4.1           | 6.4            | None    |
+---------+-----------------+----------------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Connection method: Cluster object,Cluster type: dask_gateway.GatewayCluster
Dashboard: /services/dask-gateway/clusters/ws-jupyterhub.b1a011d989384f12b3b6c3ea81618554/status,


You can verify that your workers are available and check their versions:

In [7]:
# Check versions across all workers
client.get_versions(check=True)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:4677: UserWarning: Mismatched versions found

+---------+-----------------+----------------+---------+
| Package | Client          | Scheduler      | Workers |
+---------+-----------------+----------------+---------+
| python  | 3.11.11.final.0 | 3.11.9.final.0 | None    |
| tornado | 6.4.1           | 6.4            | None    |
+---------+-----------------+----------------+---------+
  warnings.warn(msg["warning"])


{'scheduler': {'host': {'python': '3.11.9.final.0',
   'python-bits': 64,
   'OS': 'Linux',
   'OS-release': '6.8.0-79-generic',
   'machine': 'x86_64',
   'processor': 'x86_64',
   'byteorder': 'little',
   'LC_ALL': 'C.UTF-8',
   'LANG': 'C.UTF-8'},
  'packages': {'python': '3.11.9.final.0',
   'dask': '2024.5.1',
   'distributed': '2024.5.1',
   'msgpack': '1.0.8',
   'cloudpickle': '3.0.0',
   'tornado': '6.4',
   'toolz': '0.12.1',
   'numpy': '1.26.4',
   'pandas': '2.2.2',
   'lz4': '4.3.3'}},
 'workers': {},
 'client': {'host': {'python': '3.11.11.final.0',
   'python-bits': 64,
   'OS': 'Linux',
   'OS-release': '6.8.0-79-generic',
   'machine': 'x86_64',
   'processor': 'x86_64',
   'byteorder': 'little',
   'LC_ALL': 'C.UTF-8',
   'LANG': 'C.UTF-8'},
  'packages': {'python': '3.11.11.final.0',
   'dask': '2024.5.1',
   'distributed': '2024.5.1',
   'msgpack': '1.0.8',
   'cloudpickle': '3.0.0',
   'tornado': '6.4.1',
   'toolz': '0.12.1',
   'numpy': '1.26.4',
   'pandas': '

## 5. Your first parallel computation

Now that we have a DASK cluster running, let's perform a simple parallel computation to see DASK in action.

### 5.1 Create a large DASK array

We'll create a large 3D array representing a dataset similar to what you might encounter in Earth observation (e.g., time-series of spatial data).

In [8]:
# Define dimensions (time, latitude, longitude)
time_steps = 365    # One year of daily data
lat_points = 500    # 500 latitude points
lon_points = 500    # 500 longitude points

# Create a large random array with DASK
# The 'chunks' parameter defines how the array is divided for parallel processing
large_array = da.random.random(
    (time_steps, lat_points, lon_points), 
    chunks=(50, 100, 100)  # Process in smaller blocks
)

print(f"Array shape: {large_array.shape}")
print(f"Array size in memory: {large_array.nbytes / 1e9:.2f} GB")
print(f"Chunk size: {large_array.chunksize}")
print(f"Number of chunks: {large_array.npartitions}")

Array shape: (365, 500, 500)
Array size in memory: 0.73 GB
Chunk size: (50, 100, 100)
Number of chunks: 200


:::{note}

Notice that creating the array was instant! This is because DASK uses **lazy evaluation** - the array hasn't actually been created yet, just the recipe for creating it.

:::

### 5.2 Perform a computation

Let's calculate the mean across the time dimension. This operation will be distributed across our workers.

In [9]:
%%time

# Calculate the mean over the time dimension
# This operation is still lazy - no computation yet!
mean_over_time = large_array.mean(axis=0)

print("Mean calculation defined (lazy operation)")
print(f"Result shape will be: {mean_over_time.shape}")

Mean calculation defined (lazy operation)
Result shape will be: (500, 500)
CPU times: user 1.14 ms, sys: 1.82 ms, total: 2.96 ms
Wall time: 2.71 ms


In [10]:
%%time

# Now we actually compute the result
# This triggers the parallel execution across workers
result = mean_over_time.compute()

print(f"\nComputation complete!")
print(f"Result shape: {result.shape}")
print(f"Result sample value: {result[0, 0]:.6f}")


Computation complete!
Result shape: (500, 500)
Result sample value: 0.508551
CPU times: user 36.8 ms, sys: 12.9 ms, total: 49.8 ms
Wall time: 12.3 s


:::{note}

Watch the DASK dashboard while the computation runs! You'll see:
- Tasks being distributed to workers
- Memory usage increasing as data is processed
- Progress bars showing task completion

:::

### 5.3 Working with xarray and DASK

DASK integrates seamlessly with xarray, which is commonly used for Earth observation data. Let's create a more realistic example:

In [11]:
# Create coordinate arrays
time = np.arange(0, 365)
lat = np.linspace(-90, 90, 500)
lon = np.linspace(-180, 180, 500)

# Create a DASK-backed xarray DataArray
data_dask = da.random.random((365, 500, 500), chunks=(50, 100, 100))

ds = xr.DataArray(
    data_dask,
    dims=("time", "lat", "lon"),
    coords={"time": time, "lat": lat, "lon": lon},
    name="temperature"
)

print("Created xarray DataArray with DASK backend:")
display(ds)

Created xarray DataArray with DASK backend:


<xarray.DataArray 'temperature' (time: 365, lat: 500, lon: 500)> Size: 730MB
dask.array<random_sample, shape=(365, 500, 500), dtype=float64, chunksize=(50, 100, 100), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) int64 3kB 0 1 2 3 4 5 6 7 ... 358 359 360 361 362 363 364
  * lat      (lat) float64 4kB -90.0 -89.64 -89.28 -88.92 ... 89.28 89.64 90.0
  * lon      (lon) float64 4kB -180.0 -179.3 -178.6 -177.8 ... 178.6 179.3 180.0

In [12]:
%%time

# Perform a realistic analysis: select a region and compute statistics
# Select a region (e.g., Europe)
europe = ds.sel(lat=slice(35, 70), lon=slice(-10, 40))

# Calculate temporal mean and standard deviation
temporal_mean = europe.mean(dim="time").compute()
temporal_std = europe.std(dim="time").compute()

print("Regional statistics computed!")
print(f"Mean value range: {float(temporal_mean.min()):.4f} to {float(temporal_mean.max()):.4f}")
print(f"Std dev range: {float(temporal_std.min()):.4f} to {float(temporal_std.max()):.4f}")

Regional statistics computed!
Mean value range: 0.4457 to 0.5526
Std dev range: 0.2655 to 0.3137
CPU times: user 38.7 ms, sys: 5 ms, total: 43.7 ms
Wall time: 603 ms


## 6. Managing your DASK cluster

### 6.1 Scaling your cluster

You can dynamically adjust the number of workers in your cluster based on your computational needs.

In [13]:
# Scale up to 4 workers
print("Scaling cluster to 4 workers...")
cluster.scale(4)

# Wait a moment for workers to start
import time
time.sleep(5)

# Check current worker count
print(f"Current workers: {len(client.scheduler_info()['workers'])}")

Scaling cluster to 4 workers...


/opt/conda/lib/python3.11/site-packages/dask_gateway/client.py:675: GatewayWarning: Scale request of 4 workers would exceed resource limit of 3 workers. Scaling to 3 instead.
  warnings.warn(GatewayWarning(msg["msg"]))


Current workers: 3


### 6.2 Monitoring cluster resources

Check the status of your cluster and workers:

In [14]:
# Get information about the scheduler and workers
scheduler_info = client.scheduler_info()

print(f"Cluster status:")
print(f"  Total workers: {len(scheduler_info['workers'])}")
print(f"  Total cores: {sum(w['nthreads'] for w in scheduler_info['workers'].values())}")
print(f"  Total memory: {sum(w['memory_limit'] for w in scheduler_info['workers'].values()) / 1e9:.2f} GB")

print(f"\nWorker details:")
for worker_id, worker_info in list(scheduler_info['workers'].items())[:2]:  # Show first 2
    print(f"  {worker_id}:")
    print(f"    Cores: {worker_info['nthreads']}")
    print(f"    Memory: {worker_info['memory_limit'] / 1e9:.2f} GB")

Cluster status:
  Total workers: 3
  Total cores: 3
  Total memory: 3.22 GB

Worker details:
  tls://10.42.41.125:41271:
    Cores: 1
    Memory: 1.07 GB
  tls://10.42.43.137:34883:
    Cores: 1
    Memory: 1.07 GB


### 6.3 Closing your cluster

When you're done with your computations, it's important to close your cluster to free up resources.

In [15]:
# Close the client connection
client.close()
print("Client closed")

# Shutdown the cluster
cluster.close()
print("Cluster shutdown complete")

Client closed
Cluster shutdown complete


:::{warning}

Always remember to close your cluster when you're finished! Leaving clusters running unnecessarily consumes resources and may impact other users.

:::

## 7. Take home messages

**Key takeaways from this notebook:**

1. **DASK enables parallel computing** - It allows you to scale your Python data analysis workflows beyond a single machine

2. **Use DASK for large datasets** - When your data doesn't fit in memory or when you need to speed up computations through parallelization

3. **WEkEO DASK Gateway provides managed clusters** - Easy-to-use interface for creating and managing DASK clusters without infrastructure management

4. **Lazy evaluation is powerful** - DASK doesn't compute until you call `.compute()`, allowing it to optimize the entire computation

5. **Monitor with the dashboard** - The DASK dashboard provides valuable insights into cluster performance and task execution

6. **Scale as needed** - Adjust your cluster size based on your computational requirements

7. **Clean up resources** - Always close your cluster when finished to free up resources

**Best practices:**

- Start with a small cluster and scale up if needed
- Use the dashboard to monitor performance and identify bottlenecks
- Choose appropriate chunk sizes for your data
- Close clusters when you're done working

## 8. Next steps

Now that you understand the basics of DASK and the WEkEO DASK Gateway, you can:

- **Explore performance comparisons**: Check out [03_Performance_benchmark_of_DASK_Gateway.ipynb](./03_Performance_benchmark_of_DASK_Gateway.ipynb) to see detailed performance benchmarks
- **Apply DASK to real data**: Try using DASK with actual Copernicus data from the WEkEO Data Store
- **Learn more about DASK**: Visit the [official DASK tutorial](https://tutorial.dask.org/) for more advanced topics
- **Optimize your workflows**: Experiment with different chunk sizes and worker configurations for your specific use cases

### Additional resources:

- [DASK Documentation](https://docs.dask.org/)
- [DASK Best Practices](https://docs.dask.org/en/latest/best-practices.html)
- [xarray with DASK](https://docs.xarray.dev/en/stable/user-guide/dask.html)
- [WEkEO Help Center](https://help.wekeo.eu/)

<hr>

[View on GitHub](https://github.com/wekeo) | 
[Visit WEkEO](https://wekeo.copernicus.eu) | 
[Run Notebooks on the WEkEO Workspace](https://jupyterhub.prod.wekeo2.eu/) | 
[Contact helpdesk for support](mailto:support@wekeo.eu)